In [3]:
import os
import glob
import random
import subprocess
import sys
import shutil
import numpy as np
import rasterio
import torch
import albumentations as A
from torch import nn
from torch.utils.data import Dataset
from torchvision.transforms import functional as F
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)

# --- Experiment Configuration ---
class Config:
    # Experiment Identifiers
    EXPERIMENT_NAME = "segformer_b0_cems_fire"
    DATA_ROOT = "./extracted_data"

    # Data Parameters
    INPUT_BANDS = [11, 7, 3] # Sentinel-2 Indices (0-based): B12, B8, B4
    CROP_SIZE = (512, 512)

    # Model Architecture
    MODEL_CHECKPOINT = "nvidia/mit-b0"
    NUM_LABELS = 2
    ID2LABEL = {0: "Safe", 1: "Fire"}
    LABEL2ID = {"Safe": 0, "Fire": 1}

    # Training Hyperparameters
    BATCH_SIZE = 4
    LEARNING_RATE = 6e-5
    EPOCHS = 15
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Set Reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

print(f"⚙️ Configuration loaded for: {Config.EXPERIMENT_NAME}")
print(f"🔧 Device: {Config.DEVICE}")

⚙️ Configuration loaded for: segformer_b0_cems_fire
🔧 Device: cuda


In [4]:
class DataManager:
    """
    Handles automated dataset downloading and extraction.
    Compatible with both local environments and cloud instances.
    """
    DATASET_URL = "https://huggingface.co/datasets/links-ads/wildfires-cems"
    REPO_DIR = "wildfires-cems"
    EXTRACT_DIR = Config.DATA_ROOT

    @staticmethod
    def _run_shell(command):
        try:
            subprocess.check_call(command, shell=True)
        except subprocess.CalledProcessError:
            print(f"❌ Error executing command: {command}")
            sys.exit(1)

    @classmethod
    def setup_dataset(cls):
        # 1. Smart Check: If data exists, skip EVERYTHING.
        if os.path.exists(cls.EXTRACT_DIR) and os.path.exists(os.path.join(cls.EXTRACT_DIR, "train")):
            print(f"✅ Dataset already available in '{cls.EXTRACT_DIR}'. Skipping download.")
            return

        print("🔄 Initializing dataset setup...")

        # 2. Clone Repository
        if not os.path.exists(cls.REPO_DIR):
            print(f"⬇️ Cloning repository from {cls.DATASET_URL}...")
            cls._run_shell(f"GIT_LFS_SKIP_SMUDGE=1 git clone {cls.DATASET_URL}")

        # 3. Pull LFS Files
        if not os.path.exists(os.path.join(cls.REPO_DIR, "data/train/train.tar.aa")):
             print("⬇️ Downloading large files (LFS)...")
             cls._run_shell(f"cd {cls.REPO_DIR} && git lfs pull")

        # 4. Extract Archives
        print("📦 Extracting dataset archives...")
        os.makedirs(cls.EXTRACT_DIR, exist_ok=True)

        splits = ['train', 'val']
        if os.path.exists(f"{cls.REPO_DIR}/data/test"): splits.append('test')

        for split in splits:
            print(f"   - Processing {split} set...")
            cls._run_shell(f"cat {cls.REPO_DIR}/data/{split}/{split}.tar.* | tar -xzv - -i -C {cls.EXTRACT_DIR}")

        print("✅ Dataset setup complete.")

# Trigger Setup
DataManager.setup_dataset()

🔄 Initializing dataset setup...
⬇️ Downloading large files (LFS)...
📦 Extracting dataset archives...
   - Processing train set...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



❌ Error executing command: cat wildfires-cems/data/train/train.tar.* | tar -xzv - -i -C ./extracted_data
Traceback (most recent call last):
  File "/tmp/ipython-input-3874169120.py", line 13, in _run_shell
    subprocess.check_call(command, shell=True)
  File "/usr/lib/python3.12/subprocess.py", line 413, in check_call
    raise CalledProcessError(retcode, cmd)
subprocess.CalledProcessError: Command 'cat wildfires-cems/data/train/train.tar.* | tar -xzv - -i -C ./extracted_data' returned non-zero exit status 2.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-3874169120.py", line 51, in <cell line: 0>
    DataManager.setup_dataset()
  File "/tmp/ipython-input-3874169120.py", line 46, in setup_dataset
    cls._run_shell(f"cat {cls.REPO_DIR}/data/{sp

TypeError: object of type 'NoneType' has no len()

In [ ]:
class FireSegmentationDataset(Dataset):
    """
    Custom Dataset for Sentinel-2 Wildfire Segmentation.
    """
    def __init__(self, root_dir, processor, config, split="train"):
        self.root_dir = os.path.join(root_dir, split)
        self.processor = processor
        self.config = config
        self.bands = config.INPUT_BANDS

        self.image_paths = glob.glob(os.path.join(self.root_dir, "**", "*_S2L2A*.tif"), recursive=True)
        print(f"📂 [{split.upper()}] Loaded {len(self.image_paths)} samples.")

        # Data Augmentation Pipeline
        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.2, brightness_limit=0.1, contrast_limit=0.1),
        ])

    def _normalize(self, image):
        # Normalize Sentinel-2 16-bit integers to [0, 1] float
        if image.max() > 255: return np.clip(image / 10000.0, 0, 1.0)
        elif image.max() > 1: return np.clip(image / 255.0, 0, 1.0)
        return np.clip(image, 0, 1.0)

    def _smart_crop(self, img_t, msk_t):
        # Centers crop on positive class (Fire) if present
        _, h, w = img_t.shape
        crop_h, crop_w = self.config.CROP_SIZE

        if h <= crop_h or w <= crop_w: return img_t, msk_t

        fire_indices = torch.nonzero(msk_t.squeeze())

        # 80% chance to focus on fire if it exists
        if len(fire_indices) > 0 and random.random() < 0.8:
            idx = random.randint(0, len(fire_indices) - 1)
            center_y, center_x = fire_indices[idx]
            top = max(0, min(int(center_y) - crop_h // 2, h - crop_h))
            left = max(0, min(int(center_x) - crop_w // 2, w - crop_w))
        else:
            top = random.randint(0, h - crop_h)
            left = random.randint(0, w - crop_w)

        return F.crop(img_t, top, left, crop_h, crop_w), F.crop(msk_t, top, left, crop_h, crop_w)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img_path = self.image_paths[idx]

            # Locate corresponding mask
            mask_path = img_path.replace("_S2L2A", "_DEL")
            if not os.path.exists(mask_path): mask_path = img_path.replace("_S2L2A", "_GRA")

            # Load Image
            with rasterio.open(img_path) as src:
                image = src.read([b + 1 for b in self.bands]).astype(np.float32)
                image = self._normalize(image)

            # Load Mask
            with rasterio.open(mask_path) as src:
                mask = src.read(1).astype(np.float32)
                mask[mask > 0] = 1.0

            # Transform & Crop
            img_t, msk_t = self._smart_crop(torch.tensor(image), torch.tensor(mask).unsqueeze(0))

            # Augment
            img_np = img_t.permute(1, 2, 0).numpy()
            msk_np = msk_t.squeeze().numpy()
            if self.aug:
                augmented = self.aug(image=img_np, mask=msk_np)
                img_np, msk_np = augmented['image'], augmented['mask']

            # Process for Model
            encoded = self.processor(img_np, msk_np, return_tensors="pt", do_resize=False, do_rescale=False)

            return {
                "pixel_values": encoded.pixel_values.squeeze(),
                "labels": encoded.labels.squeeze().long()
            }
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

In [ ]:
class HybridLoss(nn.Module):
    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth
        self.ce = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)

        # Dice Loss calculation
        probs = torch.softmax(inputs, dim=1)
        fire_probs = probs[:, 1].contiguous().view(-1) # Class 1 = Fire
        targets_flat = targets.contiguous().view(-1)
        intersection = (fire_probs * targets_flat).sum()
        dice_score = (2. * intersection + self.smooth) / (fire_probs.sum() + targets_flat.sum() + self.smooth)

        return 0.5 * ce_loss + 0.5 * (1 - dice_score)

class SegmentationTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = HybridLoss().to(Config.DEVICE)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Upsample logits to match ground truth resolution (SegFormer outputs 1/4 res)
        upsampled_logits = nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        )

        loss = self.loss_fn(upsampled_logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# 1. Initialize Processor & Model
print(f"🚀 Loading Model: {Config.MODEL_CHECKPOINT}")
processor = SegformerImageProcessor.from_pretrained(
    Config.MODEL_CHECKPOINT,
    do_reduce_labels=False,
    do_resize=False,
    do_rescale=False
)

model = SegformerForSemanticSegmentation.from_pretrained(
    Config.MODEL_CHECKPOINT,
    num_labels=Config.NUM_LABELS,
    id2label=Config.ID2LABEL,
    label2id=Config.LABEL2ID,
    ignore_mismatched_sizes=True
)

# 2. Prepare Datasets
train_ds = FireSegmentationDataset(Config.DATA_ROOT, processor, Config, split="train")
val_ds = FireSegmentationDataset(Config.DATA_ROOT, processor, Config, split="val")

# 3. Define Arguments
args = TrainingArguments(
    output_dir=f"./results/{Config.EXPERIMENT_NAME}",
    learning_rate=Config.LEARNING_RATE,
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    logging_steps=50,
    fp16=(Config.DEVICE == "cuda"),
    remove_unused_columns=False,
    report_to="none"
)

# 4. Train
trainer = SegmentationTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print(f"🔥 Starting training on {Config.DEVICE}...")
trainer.train()

# 5. Save
save_path = f"./models/{Config.EXPERIMENT_NAME}"
trainer.save_model(save_path)
processor.save_pretrained(save_path)
print(f"🏆 Training finished. Model saved to {save_path}")